# 05 — Extreme Gradient Boosting (XGBoost) Walk-Forward Signals
**GeoSentinel Terminal (VARTA) · Team 7 Lambda · SP2026**

Builds regime-dependent return prediction signals using Extreme Gradient Boosting (XGBoost)
with strict walk-forward (out-of-sample) validation.

- **Training window per fold:** 36 months
- **Test window per fold:** 3 months
- **Total folds:** ~50 across 14 years
- **Rule:** Out-of-Sample (OOS) metrics ONLY — never report in-sample accuracy anywhere

Features: lagged returns (1d, 5d, 21d), rolling volatility, Geopolitical Risk Index (GPR),
Global Supply Chain Pressure Index (GSCPI), regime label.

Outputs: `data/processed/signals.parquet`

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [2]:
import numpy as np
import pandas as pd
import polars as pl
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from config import (
    DATA_PROC, TICKERS,
    WF_TRAIN_MONTHS, WF_TEST_MONTHS,
    XGB_N_ESTIMATORS, XGB_MAX_DEPTH, XGB_LEARNING_RATE, XGB_SUBSAMPLE,
    REGIME_CRISIS,
)
from src.utils import log, save_parquet

log.info("Loading processed data...")

2026-04-16 08:48:25 [INFO] varta — Loading processed data...


In [3]:
# ── Load all inputs ────────────────────────────────────────────────────────────
prices  = pl.read_parquet(DATA_PROC / "prices.parquet")
fred    = pl.read_parquet(DATA_PROC / "fred.parquet")
regimes = pl.read_parquet(DATA_PROC / "regimes.parquet")

# Normalise date column to Date type across all frames to avoid datetime precision mismatches
prices  = prices.with_columns(pl.col("date").cast(pl.Date))
fred    = fred.with_columns(pl.col("date").cast(pl.Date))
regimes = regimes.with_columns(pl.col("date").cast(pl.Date))

# Pivot FRED to wide format: one row per date
fred_wide = (
    fred.pivot(index="date", on="series_id", values="value")
    .sort("date")
)
log.info(f"Prices: {prices.shape} | FRED wide: {fred_wide.shape} | Regimes: {regimes.shape}")

2026-04-16 08:48:25 [INFO] varta — Prices: (43474, 8) | FRED wide: (5267, 6) | Regimes: (3629, 4)


In [4]:
# ── Build feature matrix for each ticker ──────────────────────────────────────
def build_features_for_ticker(ticker: str) -> pd.DataFrame:
    """Build lagged feature matrix for one ticker, joined with macro signals."""
    df_t = (
        prices.filter(pl.col("ticker") == ticker)
        .sort("date")
        .with_columns([
            pl.col("return_1d").shift(1).alias("lag_1d"),
            pl.col("return_1d").shift(5).alias("lag_5d"),
            pl.col("return_1d").shift(21).alias("lag_21d"),
            pl.col("return_1d").rolling_std(window_size=21).alias("vol_21d"),
            # Target: 1 if next month (21d) forward return > 0, else 0
            pl.col("close").shift(-21).truediv(pl.col("close")).sub(1)
              .map_elements(lambda x: 1 if x is not None and x > 0 else 0, return_dtype=pl.Int32)
              .alias("target"),
        ])
        .join(fred_wide.select(["date", "GPR", "GSCPI", "OIL_BRENT"]), on="date", how="left")
        .join(regimes.select(["date", "regime_label"]), on="date", how="left")
        .with_columns([
            (pl.col("regime_label") == REGIME_CRISIS).cast(pl.Int32).alias("is_crisis")
        ])
        .drop_nulls(["lag_1d", "lag_5d", "lag_21d", "vol_21d", "GPR", "GSCPI", "target"])
    )
    df_pd = df_t.to_pandas()
    df_pd["ticker"] = ticker
    return df_pd

# Build for all tickers
all_frames = [build_features_for_ticker(t) for t in TICKERS]
feature_pd = pd.concat(all_frames, ignore_index=True)
print(f"Feature matrix: {feature_pd.shape}")
feature_pd.head(3)

Feature matrix: (42514, 18)


,date,ticker,open,high,low,close,volume,return_1d,lag_1d,lag_5d,lag_21d,vol_21d,target,GPR,GSCPI,OIL_BRENT,regime_label,is_crisis
0,2010-11-30,REMX,152.259247,153.408665,151.492972,152.795639,21383.0,-0.011403,0.000000,0.011460,0.056382,0.021209,1,183.407700,-0.522588,86.02,Normal,0
1,2010-12-01,REMX,158.312825,158.312825,155.860741,157.853058,9833.0,0.033099,-0.011403,-0.033498,-0.005337,0.022469,1,94.047661,-0.468022,88.56,Normal,0
2,2010-12-02,REMX,158.542737,160.535054,158.006345,159.845413,23400.0,0.012622,0.033099,0.024975,-0.002439,0.022612,1,100.178459,-0.466106,89.37,Normal,0


In [5]:
# ── Walk-forward validation loop ──────────────────────────────────────────────
FEATURE_COLS = ["lag_1d", "lag_5d", "lag_21d", "vol_21d", "GPR", "GSCPI", "OIL_BRENT", "is_crisis"]

results = []
predictions_all = []

for ticker in TICKERS:
    df_t = feature_pd[feature_pd["ticker"] == ticker].copy().sort_values("date").reset_index(drop=True)
    df_t["date"] = pd.to_datetime(df_t["date"])
    df_t = df_t.set_index("date")

    dates = df_t.index
    start_date = dates.min()
    end_date   = dates.max()

    fold = 0
    train_end = start_date + pd.DateOffset(months=WF_TRAIN_MONTHS)

    while train_end + pd.DateOffset(months=WF_TEST_MONTHS) <= end_date:
        test_start = train_end
        test_end   = train_end + pd.DateOffset(months=WF_TEST_MONTHS)

        train = df_t.loc[start_date:train_end]
        test  = df_t.loc[test_start:test_end]

        if len(train) < 100 or len(test) < 10:
            train_end += pd.DateOffset(months=WF_TEST_MONTHS)
            continue

        X_train, y_train = train[FEATURE_COLS].values, train["target"].values
        X_test,  y_test  = test[FEATURE_COLS].values,  test["target"].values

        model = XGBClassifier(
            n_estimators=XGB_N_ESTIMATORS, max_depth=XGB_MAX_DEPTH,
            learning_rate=XGB_LEARNING_RATE, subsample=XGB_SUBSAMPLE,
            eval_metric="logloss", verbosity=0, random_state=42,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        # Out-of-Sample metrics ONLY
        results.append({
            "ticker":        ticker,
            "fold":          fold,
            "test_start":    test_start,
            "test_end":      test_end,
            "oos_accuracy":  accuracy_score(y_test, y_pred),
            "oos_precision": precision_score(y_test, y_pred, zero_division=0),
            "oos_recall":    recall_score(y_test, y_pred, zero_division=0),
            "n_test":        len(test),
        })
        for i, (idx, row) in enumerate(test.iterrows()):
            predictions_all.append({
                "date": idx, "ticker": ticker,
                "predicted_direction": int(y_pred[i]),
                "confidence": float(y_prob[i]),
                "actual_direction": int(y_test[i]),
                "regime": row.get("regime_label", None),
            })
        fold += 1
        train_end += pd.DateOffset(months=WF_TEST_MONTHS)

results_df  = pd.DataFrame(results)
preds_df    = pd.DataFrame(predictions_all)
log.info(f"Walk-forward complete: {len(results_df)} folds across {len(TICKERS)} tickers")

2026-04-16 08:50:08 [INFO] varta — Walk-forward complete: 527 folds across 12 tickers


In [6]:
# ── Out-of-Sample summary ─────────────────────────────────────────────────────
print("=== Out-of-Sample Metrics Summary (never in-sample) ===")
print(results_df.groupby("ticker")[["oos_accuracy", "oos_precision", "oos_recall"]].mean().round(3))
print(f"\nOverall mean OOS accuracy: {results_df['oos_accuracy'].mean():.3f}")

=== Out-of-Sample Metrics Summary (never in-sample) ===
        oos_accuracy  oos_precision  oos_recall
ticker                                         
ALB            0.490          0.517       0.528
AMD            0.522          0.558       0.633
BNO            0.493          0.562       0.549
CVX            0.538          0.590       0.693
FCX            0.520          0.514       0.445
GLD            0.503          0.564       0.520
LIT            0.558          0.594       0.511
NVDA           0.602          0.671       0.778
REMX           0.531          0.535       0.304
SPY            0.620          0.699       0.810
TSM            0.556          0.640       0.678
XOM            0.483          0.544       0.629

Overall mean OOS accuracy: 0.535


In [7]:
# ── Save ──────────────────────────────────────────────────────────────────────
signals_pl = pl.from_pandas(preds_df)
oos_pl     = pl.from_pandas(results_df)

save_parquet(signals_pl, DATA_PROC / "signals.parquet",     "XGBoost predictions")
save_parquet(oos_pl,     DATA_PROC / "oos_metrics.parquet", "OOS metrics per fold")
print("Saved → data/processed/signals.parquet + oos_metrics.parquet")

2026-04-16 08:50:08 [INFO] varta — Saved XGBoost predictions → /Users/taruntheegela/Desktop/VARTA/data/processed/signals.parquet (33,158 rows)


2026-04-16 08:50:08 [INFO] varta — Saved OOS metrics per fold → /Users/taruntheegela/Desktop/VARTA/data/processed/oos_metrics.parquet (527 rows)


Saved → data/processed/signals.parquet + oos_metrics.parquet
